In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')


In [ ]:
import asyncio
import collections
import concurrent.futures
import json
import re
import time
import litellm
import pandas as pd
import torch
import yaml
from pathlib import Path
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import EvalConfig, build_submission_limits, ALLOWED_ADAPTER_EXTENSIONS
from swegemma.evaluate import Evaluator
from swegemma.models import load_tasks

litellm.drop_params = True

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
DECLARED_MODEL = 'gemma-4-e4b-it-qat-w4a16-ct'
MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{DECLARED_MODEL}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}'

SOUND12_IDS = ["fastapi_14297", "fastapi_14301", "fastapi_14361", "fastapi_14372", "fastapi_14458", "fastapi_14463", "fastapi_14786", "requests_6644", "requests_7205", "requests_7309", "requests_7427", "rich_3006", "rich_3296", "rich_3518", "rich_3894", "rich_3905"]
VARIANTS = {
  "v9a_think1024_verify_v6": {
    "agent.yaml": "name: gemma4_e4b_v9a_think1024_verify_v6\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: Single-agent E4B think=1024 private + verify-patch v6 (baseline-diff pytest, CLI exclusion, syntax context, loop guard).\ninstruction: !include prompts/system.md\nskills:\n  - skills/verify-patch\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\ngenerate_content_config: !include configs/sampling.yaml\n",
    "configs/sampling.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 1024\n  include_thoughts: false\n",
    "eval_config.yaml": "evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 60\n  max_time_minutes: 8\n  max_turns: 120\n",
    "prompts/system.md": "You are an autonomous software engineer fixing an issue in `/workspace`.\nYour core tools are `run_command`, `read_file`, `edit_file`, `write_file`, `get_status`, `submit_patch`, plus the `verify-patch` skill (`run_skill_script`). Always run shell commands via `run_command(command=...)`.\nGoal: make the smallest correct edit to SOURCE code in `/workspace` so the hidden test suite passes, verify with real tests + the `verify-patch` skill, then call `submit_patch`.\n\n## Workflow (Budget: 60 tool calls, 8 minutes)\n1. **Locate source fast (calls 1\u20136)**:\n   - Call `run_command` with `git grep -n -F -e '<Symbol1>' -e '<Symbol2>' -- '*.py' | head -25`.\n   - Call `read_file` on 80\u2013120 lines around the target implementation. For FastAPI doc-code issues, edit executable files under `docs_src/`.\n2. **Edit cleanly (by call 12 \u2014 reliable editing rules)**:\n   - Copy `old_string` **verbatim** from fresh `read_file` or `sed -n 'A,Bp' <file>` output (short 3\u201312 lines, exact indentation, NO manual `\\n`/`\\\"` escapes, `allow_multiple=false`).\n   - **After ANY single failed `edit_file` (including `old_string not found` OR missing parameter errors)**, DO NOT retry `edit_file` \u2014 immediately switch to a single-quoted Python heredoc via `run_command`:\n     ```bash\n     python3 - <<'EOF'\n     from pathlib import Path\n     p = Path(\"path/to/file.py\")\n     s = p.read_text()\n     old = \"\"\"exact old block\"\"\"\n     new = \"\"\"exact new block\"\"\"\n     assert s.count(old) == 1, f\"count={s.count(old)}\"\n     p.write_text(s.replace(old, new, 1))\n     EOF\n     ```\n   - NEVER use multiline `python3 -c \"...\"` (quotes break). ALWAYS use `python3 - <<'EOF' ... EOF`.\n   - NEVER call `write_file` on `/tmp/...` or to create scratch/test scripts.\n3. **Verify with `verify-patch` + real bug repro (REQUIRED before `submit_patch`)**:\n   - First, run an inline bug repro + 2 edge cases importing the real `/workspace` module via `python3 - <<'EOF' ... EOF` to confirm the reported bug is truly fixed.\n   - Next, call `run_skill_script(skill_name=\"verify-patch\", file_path=\"scripts/run_verify.py\")`. It automatically finds the REAL matching files in `tests/` (never guess test paths!), checks syntax/diff/protected files, and runs targeted `pytest`.\n   - **If `verify-patch` returns `VERDICT: FAIL` (or your bug repro fails)**: DO NOT call `submit_patch`. Read the traceback, fix the source code, and re-run your repro + `verify-patch` until `VERDICT: PASS`.\n   - If `verify-patch` reports a `syntax_error` you cannot fix in 1 edit, reset cleanly with `git checkout -- <file>` and re-apply your edit.\n   - NEVER call `verify-patch` twice in a row without editing source code in between (if `ok: true`, call `submit_patch`).\n4. **Submit cleanly**:\n   - Only after `verify-patch` reports `ok: true` (`VERDICT: PASS`), call `submit_patch` as your final tool action, confirm `patch_size > 0` and `files_changed > 0`, and write a 1-sentence summary.\n\n## Hard Rules\n- NEVER modify test files (`tests/`, `test_*.py`, `*_test.py`) or configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg`, `tox.ini`).\n- NEVER guess `pytest tests/...` paths without running `verify-patch` or `find tests -name ...`.\n- NEVER run bare `pytest` (`pytest .`), `pip install`, or `git commit`.\n- NEVER call `submit_patch` when `verify-patch` or your repro script just reported a failure (unless `budget_warning` forces an immediate final submit).\n- NEVER finish without calling `submit_patch` on a compiled, non-empty source fix.\n",
    "skills/verify-patch/SKILL.md": "---\nname: verify-patch\ndescription: Deterministic pre-submit verification gate that inspects git diff, checks Python syntax and protected files, auto-discovers matching real test files in tests/, runs baseline-diff targeted pytest (filtering pre-existing sandbox failures), and outputs a clear PASS/FAIL verdict.\n---\n\n# Verify Patch Skill\n\nRun `scripts/run_verify.py` via `run_skill_script(skill_name=\"verify-patch\", file_path=\"scripts/run_verify.py\")` after a source edit and right before `submit_patch`.\n\nIt deterministically verifies:\n1. Non-empty diff on source files (no protected test/config files modified, no stray untracked files).\n2. Zero Python syntax errors (`ast.parse`) across modified `.py` files (with line context & reset command if broken).\n3. Auto-discovers real `tests/` files matching your modified source files/symbols (skipping unrelated tutorial/CLI tests).\n4. Runs baseline-diff targeted `pytest`: filters out pre-existing sandbox failures so `FAIL` only triggers on real regressions introduced by your patch.\n",
    "skills/verify-patch/scripts/run_verify.py": "\"\"\"Deterministic pre-submit patch & baseline-diff targeted pytest verifier (v6).\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nPROTECTED_CONFIGS = {\n    \"conftest.py\", \"pytest.ini\", \"pyproject.toml\", \"tox.ini\",\n    \"setup.cfg\", \".pytest.ini\", \"sitecustomize.py\", \"usercustomize.py\",\n    \"_swegemma_stubs.py\",\n}\n\nGENERIC_DIRS = {\"src\", \"lib\", \"pkg\", \"docs_src\", \"\"}\nGENERIC_STEMS = {\"init\", \"main\", \"base\", \"types\", \"typing\"}\n\n\ndef is_cache_artifact(rel_path: str) -> bool:\n    p = Path(rel_path)\n    parts = set(p.parts)\n    if parts & {\"__pycache__\", \".pytest_cache\", \".mypy_cache\", \".ruff_cache\"}:\n        return True\n    if p.suffix in {\".pyc\", \".pyo\"}:\n        return True\n    return False\n\n\ndef is_protected(rel_path: str) -> bool:\n    p = Path(rel_path)\n    name = p.name\n    if name in PROTECTED_CONFIGS or p.suffix == \".pth\":\n        return True\n    parts_l = {part.lower() for part in p.parts}\n    if parts_l & {\"tests\", \"test\", \"testing\"} and p.suffix == \".py\":\n        return True\n    if name.startswith(\"test_\") and name.endswith(\".py\"):\n        return True\n    if name.endswith(\"_test.py\"):\n        return True\n    return False\n\n\ndef _run(\n    root: Path,\n    cmd: list[str],\n    timeout: int = 30,\n    extra_env: dict[str, str] | None = None,\n    text: bool = True,\n) -> subprocess.CompletedProcess:\n    env = None\n    if extra_env:\n        env = dict(os.environ)\n        env.update(extra_env)\n    return subprocess.run(\n        cmd,\n        cwd=root,\n        capture_output=True,\n        text=text,\n        timeout=timeout,\n        env=env,\n        check=False,\n    )\n\n\ndef _git_out(root: Path, *args: str) -> str:\n    return _run(root, [\"git\", \"-c\", \"safe.directory=*\", *args], timeout=15).stdout.strip()\n\n\ndef _resolve_repo_root(explicit_root: str | None) -> Path:\n    candidates: list[Path] = []\n    if explicit_root:\n        candidates.append(Path(explicit_root))\n    cwd = Path.cwd()\n    for anc in (cwd, *cwd.parents):\n        candidates.append(anc)\n        candidates.append(anc / \"workspace\")\n    candidates.append(Path(\"/workspace\"))\n\n    for cand in candidates:\n        if cand.is_dir() and (cand / \".git\").exists():\n            return cand\n    return cwd if cwd.is_dir() else Path(\"/workspace\")\n\n\ndef _resolve_pytest_cmd(root: Path) -> list[str]:\n    \"\"\"Prefer the sandbox's isolated venv (sibling to workspace/) so editable repo imports resolve.\"\"\"\n    venv_dirs = [root.parent / \"venv\", root / \".venv\", root / \"venv\"]\n    for vd in venv_dirs:\n        py_bin = vd / \"bin\" / \"pytest\"\n        if py_bin.is_file():\n            return [str(py_bin)]\n        py_exe = vd / \"bin\" / \"python\"\n        if py_exe.is_file():\n            return [str(py_exe), \"-m\", \"pytest\"]\n    pytest_bin = shutil.which(\"pytest\")\n    if pytest_bin:\n        return [pytest_bin]\n    return [sys.executable, \"-m\", \"pytest\"]\n\n\ndef _cache_path(root: Path) -> Path:\n    digest = hashlib.sha1(str(root.resolve()).encode(\"utf-8\", errors=\"ignore\")).hexdigest()[:12]\n    return Path(f\"/tmp/_swegemma_verify_cache_{digest}.json\")\n\n\ndef _load_cache(root: Path) -> dict:\n    p = _cache_path(root)\n    if p.is_file():\n        try:\n            data = json.loads(p.read_text(encoding=\"utf-8\"))\n            if isinstance(data, dict):\n                return data\n        except Exception:\n            pass\n    return {\"preexisting_failed\": [], \"last_diff_hash\": \"\", \"same_diff_streak\": 0}\n\n\ndef _save_cache(root: Path, data: dict) -> None:\n    try:\n        _cache_path(root).write_text(json.dumps(data), encoding=\"utf-8\")\n    except Exception:\n        pass\n\n\ndef _extract_changed_symbols(diff_text: str) -> list[str]:\n    syms: list[str] = []\n    seen: set[str] = set()\n    for line in diff_text.splitlines():\n        if line.startswith(\"@@\"):\n            m = re.search(r\"@@\\s+(?:def|async def|class)\\s+([A-Za-z_][A-Za-z0-9_]*)\", line)\n            if m and m.group(1) not in seen and len(m.group(1)) >= 4:\n                seen.add(m.group(1))\n                syms.append(m.group(1))\n        elif line.startswith((\"+\", \"-\")) and not line.startswith((\"+++\", \"---\")):\n            m = re.match(r\"^[+-]\\s*(?:def|async def|class)\\s+([A-Za-z_][A-Za-z0-9_]*)\", line)\n            if m and m.group(1) not in seen and len(m.group(1)) >= 4:\n                seen.add(m.group(1))\n                syms.append(m.group(1))\n    return syms[:8]\n\n\ndef _discover_test_files(root: Path, source_files: list[str], diff_text: str, max_files: int = 2) -> list[str]:\n    test_dirs = [d for d in (\"tests\", \"test\") if (root / d).is_dir()]\n    if not test_dirs:\n        return []\n\n    all_tests: list[Path] = []\n    for td in test_dirs:\n        for p in (root / td).rglob(\"*.py\"):\n            nm = p.name\n            if nm.startswith(\"test_\") or nm.endswith(\"_test.py\"):\n                all_tests.append(p.relative_to(root))\n\n    if not all_tests:\n        return []\n\n    stems: set[str] = set()\n    subpkg_tokens: set[str] = set()\n    is_cli_edit = any(\"cli\" in rel.lower() for rel in source_files)\n\n    for rel in source_files:\n        p = Path(rel)\n        if p.suffix != \".py\":\n            continue\n        clean_stem = p.stem.lower().lstrip(\"_\")\n        if clean_stem and clean_stem not in GENERIC_STEMS:\n            stems.add(clean_stem)\n        # Drop leading src/lib/pkg and the top-level package name itself (e.g. \"fastapi\", \"requests\")\n        # so generic tests like test_fastapi_cli.py don't match every file in the package.\n        pkg_dirs = [part.lower().lstrip(\"_\") for part in p.parts[:-1] if part.lower().lstrip(\"_\") not in GENERIC_DIRS]\n        for subpkg in pkg_dirs[1:]:\n            subpkg_tokens.add(subpkg)\n\n    sym_hits: dict[str, int] = {}\n    symbols = _extract_changed_symbols(diff_text)\n    if symbols:\n        grep_cmd = [\"git\", \"-c\", \"safe.directory=*\", \"grep\", \"-l\", \"-F\"]\n        for sym in symbols:\n            grep_cmd.extend([\"-e\", sym])\n        grep_cmd.append(\"--\")\n        grep_cmd.extend(test_dirs)\n        proc = _run(root, grep_cmd, timeout=10)\n        for line in proc.stdout.splitlines():\n            line = line.strip()\n            if line:\n                sym_hits[line] = sym_hits.get(line, 0) + 1\n\n    scored: list[tuple[int, int, str]] = []\n    for tp in all_tests:\n        rel_str = tp.as_posix()\n        name_l = tp.name.lower()\n        path_l = rel_str.lower()\n        # Skip tutorial/doc example tests & unrelated CLI tests unless editing CLI code\n        if \"tutorial\" in path_l:\n            continue\n        if not is_cli_edit and (\"_cli\" in name_l or \"cli_\" in name_l or \"/cli/\" in path_l):\n            continue\n        score = 0\n\n        for st in stems:\n            if name_l in (f\"test_{st}.py\", f\"{st}_test.py\"):\n                score += 120\n            elif f\"_{st}.\" in name_l or f\"_{st}_\" in name_l or name_l.startswith(f\"test_{st}_\"):\n                score += 70\n            elif st in name_l:\n                score += 40\n            if f\"/{st}/\" in path_l:\n                score += 30\n\n        for pt in subpkg_tokens:\n            if pt in name_l or f\"test_{pt}\" in path_l or f\"/{pt}/\" in path_l:\n                score += 35\n\n        if rel_str in sym_hits:\n            score += 45 * sym_hits[rel_str]\n\n        if score > 0:\n            scored.append((score, -len(rel_str), rel_str))\n\n    scored.sort(reverse=True)\n    return [item[2] for item in scored[:max_files]]\n\n\ndef _extract_failed_nodes(output: str) -> list[str]:\n    nodes: list[str] = []\n    seen: set[str] = set()\n    for line in output.splitlines():\n        m = re.match(r\"^FAILED\\s+(.+?)(?:\\s+-\\s+.*)?$\", line.strip())\n        if m:\n            node = m.group(1).strip()\n            if \"::\" in node and node not in seen:\n                seen.add(node)\n                nodes.append(node)\n    return nodes[:8]\n\n\ndef _check_baseline_failures(\n    root: Path,\n    base_ref: str,\n    source_touched: list[str],\n    failed_nodes: list[str],\n    pytest_cmd: list[str],\n) -> set[str]:\n    \"\"\"Run ONLY the failed node IDs against base_ref to separate pre-existing sandbox failures from new regressions.\"\"\"\n    if not failed_nodes or not source_touched:\n        return set()\n\n    saved_bytes: dict[Path, bytes] = {}\n    baseline_bytes: dict[Path, bytes] = {}\n\n    for rel in source_touched:\n        full = root / rel\n        if not full.is_file():\n            continue\n        # Only temporarily revert files that exist in base_ref\n        chk = _run(root, [\"git\", \"-c\", \"safe.directory=*\", \"cat-file\", \"-e\", f\"{base_ref}:{rel}\"], timeout=10)\n        if chk.returncode != 0:\n            continue\n        show = _run(root, [\"git\", \"-c\", \"safe.directory=*\", \"show\", f\"{base_ref}:{rel}\"], timeout=10, text=False)\n        if show.returncode != 0:\n            continue\n        saved_bytes[full] = full.read_bytes()\n        baseline_bytes[full] = show.stdout\n\n    if not saved_bytes:\n        return set()\n\n    try:\n        for full, bdata in baseline_bytes.items():\n            full.write_bytes(bdata)\n        cmd = [\n            *pytest_cmd,\n            *failed_nodes,\n            \"-q\", \"--tb=no\", \"--no-header\",\n            \"--continue-on-collection-errors\",\n            \"-o\", \"cache_dir=/tmp/pytest_cache\",\n        ]\n        proc = _run(root, cmd, timeout=25, extra_env={\"PYTHONDONTWRITEBYTECODE\": \"1\"})\n        combined = ((proc.stdout or \"\") + \"\\n\" + (proc.stderr or \"\")).strip()\n        return set(_extract_failed_nodes(combined))\n    except subprocess.TimeoutExpired:\n        return set()\n    finally:\n        for full, orig in saved_bytes.items():\n            full.write_bytes(orig)\n\n\ndef _run_targeted_pytest(\n    root: Path,\n    pytest_cmd: list[str],\n    matched_tests: list[str],\n    deselect_nodes: list[str],\n) -> tuple[int, str]:\n    flags = [\n        \"--maxfail=5\", \"-q\", \"--tb=short\", \"--no-header\",\n        \"--continue-on-collection-errors\",\n        \"-o\", \"cache_dir=/tmp/pytest_cache\",\n    ]\n    for node in deselect_nodes:\n        file_part = node.split(\"::\", 1)[0]\n        if file_part in matched_tests:\n            flags.extend([\"--deselect\", node])\n    cmd = [*pytest_cmd, *matched_tests, *flags]\n    proc = _run(root, cmd, timeout=60, extra_env={\"PYTHONDONTWRITEBYTECODE\": \"1\"})\n    combined = ((proc.stdout or \"\") + \"\\n\" + (proc.stderr or \"\")).strip()\n    return proc.returncode, combined\n\n\ndef _syntax_context(full: Path, lineno: int | None) -> str:\n    if not lineno or not full.is_file():\n        return \"\"\n    lines = full.read_text(encoding=\"utf-8\", errors=\"ignore\").splitlines()\n    start = max(1, lineno - 2)\n    end = min(len(lines), lineno + 2)\n    return \"\\n\".join(\n        f\"{'>>' if i == lineno else '  '} {i}: {lines[i - 1]}\"\n        for i in range(start, end + 1)\n    )\n\n\ndef verify_workspace(root: Path, run_pytest: bool = True) -> dict:\n    has_baseline = bool(_git_out(root, \"rev-parse\", \"--verify\", \"_swegemma_baseline\"))\n    base_ref = \"_swegemma_baseline\" if has_baseline else \"HEAD\"\n\n    changed = [\n        line for line in _git_out(root, \"diff\", \"--name-only\", base_ref).splitlines()\n        if line.strip() and not is_cache_artifact(line)\n    ]\n    untracked = [\n        line for line in _git_out(root, \"ls-files\", \"--others\", \"--exclude-standard\").splitlines()\n        if line.strip() and not is_cache_artifact(line)\n    ]\n    all_touched = sorted(set(changed) | set(untracked))\n\n    protected_touched = [p for p in all_touched if is_protected(p)]\n    source_touched = [p for p in all_touched if not is_protected(p)]\n\n    syntax_errors = []\n    for rel in source_touched:\n        full = root / rel\n        if full.suffix == \".py\" and full.is_file():\n            try:\n                ast.parse(full.read_text(encoding=\"utf-8\", errors=\"ignore\"), filename=rel)\n            except SyntaxError as exc:\n                syntax_errors.append({\n                    \"file\": rel,\n                    \"line\": exc.lineno,\n                    \"message\": exc.msg,\n                    \"context\": _syntax_context(full, exc.lineno),\n                })\n\n    diff_text = _git_out(root, \"diff\", \"-U0\", base_ref, \"--\", *source_touched) if source_touched else \"\"\n    diff_stat = _git_out(root, \"diff\", \"--stat\", base_ref)\n\n    cache = _load_cache(root)\n    preexisting_failed: list[str] = list(cache.get(\"preexisting_failed\") or [])\n    diff_hash = hashlib.sha256(diff_text.encode(\"utf-8\", errors=\"ignore\")).hexdigest()[:12]\n    same_diff_streak = (cache.get(\"same_diff_streak\", 0) + 1) if (diff_hash and diff_hash == cache.get(\"last_diff_hash\")) else 0\n\n    warnings: list[str] = []\n    if not source_touched:\n        warnings.append(\"No source files modified yet; submit_patch would be empty.\")\n    if protected_touched:\n        revert_cmd = \"git checkout -- \" + \" \".join(protected_touched)\n        warnings.append(f\"Protected test/config files touched (revert immediately with `{revert_cmd}`): {protected_touched}\")\n    if untracked:\n        warnings.append(f\"Untracked files in /workspace (remove scratch files): {untracked}\")\n    if syntax_errors:\n        bad_files = \" \".join(sorted({e[\"file\"] for e in syntax_errors}))\n        warnings.append(\n            f\"Python syntax error(s) detected. Inspect syntax_errors[].context and fix, \"\n            f\"or reset cleanly with `git checkout -- {bad_files}` and re-apply your edit.\"\n        )\n    if same_diff_streak >= 1:\n        warnings.append(\n            \"LOOP GUARD: Workspace diff has NOT changed since your previous verify-patch call. \"\n            \"Do NOT call verify-patch again without editing source code first (or call submit_patch if verdict is PASS).\"\n        )\n\n    matched_tests: list[str] = []\n    pytest_status = \"SKIPPED\"\n    pytest_summary = \"\"\n    filtered_preexisting: list[str] = []\n\n    if run_pytest and source_touched and not syntax_errors:\n        matched_tests = _discover_test_files(root, source_touched, diff_text, max_files=2)\n        if not matched_tests:\n            pytest_status = \"NO_MATCHING_TESTS\"\n            pytest_summary = \"No matching test file found under tests/; run an inline python3 heredoc repro instead.\"\n        else:\n            pytest_cmd = _resolve_pytest_cmd(root)\n            try:\n                for attempt in range(2):\n                    rc, combined = _run_targeted_pytest(root, pytest_cmd, matched_tests, preexisting_failed)\n                    has_real_failure = (\n                        \"=== FAILURES ===\" in combined\n                        or bool(re.search(r\"\\b\\d+\\s+failed\\b\", combined))\n                    )\n                    has_env_error = (\n                        \"=== ERRORS ===\" in combined\n                        or \"ERROR collecting\" in combined\n                        or \"ERROR at setup\" in combined\n                    )\n                    if rc == 0:\n                        pytest_status = \"PASS\"\n                        lines = [ln for ln in combined.splitlines() if ln.strip()]\n                        pytest_summary = \"\\n\".join(lines[-5:])\n                        break\n                    if rc == 5:\n                        pytest_status = \"NO_TESTS_COLLECTED\"\n                        pytest_summary = combined[-600:]\n                        break\n                    if \"No module named pytest\" in combined or \"disabled\" in combined.lower():\n                        pytest_status = \"UNAVAILABLE\"\n                        pytest_summary = \"pytest unavailable in sandbox; rely on real-import python3 heredoc check.\"\n                        break\n                    if has_real_failure:\n                        failed_nodes = _extract_failed_nodes(combined)\n                        base_failed = _check_baseline_failures(root, base_ref, source_touched, failed_nodes, pytest_cmd)\n                        new_regressions = [n for n in failed_nodes if n not in base_failed]\n                        if base_failed:\n                            for bf in sorted(base_failed):\n                                if bf not in preexisting_failed:\n                                    preexisting_failed.append(bf)\n                                if bf not in filtered_preexisting:\n                                    filtered_preexisting.append(bf)\n                        if not new_regressions and failed_nodes:\n                            # Every observed failure already existed on baseline!\n                            if attempt == 0:\n                                # Re-run once with --deselect so tests AFTER the pre-existing failure still execute.\n                                continue\n                            pytest_status = \"PASS_PREEXISTING_ONLY\"\n                            pytest_summary = (\n                                f\"Filtered pre-existing sandbox failure(s) that already fail on baseline: {filtered_preexisting}. \"\n                                \"No new regressions detected.\"\n                            )\n                            break\n                        pytest_status = \"FAIL\"\n                        pytest_summary = combined[-1800:]\n                        warnings.append(\n                            f\"CONFIRMED NEW REGRESSION(S) in {matched_tests}: {new_regressions or failed_nodes}. \"\n                            \"Inspect traceback and fix source before submitting.\"\n                        )\n                        break\n                    if has_env_error:\n                        pytest_status = \"ENV_SKIPPED\"\n                        pytest_summary = (\n                            \"Only sandbox collection/fixture setup errors occurred (0 test assertions failed). \"\n                            \"Rely on your inline python3 heredoc repro to verify the fix.\"\n                        )\n                        break\n                    pytest_status = \"FAIL\"\n                    pytest_summary = combined[-1800:]\n                    warnings.append(f\"Targeted pytest exited {rc} on {matched_tests}; inspect output before submitting.\")\n                    break\n            except subprocess.TimeoutExpired:\n                pytest_status = \"TIMEOUT\"\n                pytest_summary = f\"pytest timed out after 60s on {matched_tests}; run a narrower `-k` test filter.\"\n\n    cache[\"preexisting_failed\"] = preexisting_failed[:20]\n    cache[\"last_diff_hash\"] = diff_hash\n    cache[\"same_diff_streak\"] = same_diff_streak\n    _save_cache(root, cache)\n\n    if filtered_preexisting and pytest_status == \"PASS\":\n        pytest_summary = (\n            f\"{pytest_summary}\\n(Note: deselected pre-existing baseline failure(s): {filtered_preexisting})\"\n        ).strip()\n\n    static_ok = bool(source_touched) and not protected_touched and not syntax_errors\n    tests_ok = pytest_status in (\n        \"PASS\", \"PASS_PREEXISTING_ONLY\", \"ENV_SKIPPED\", \"NO_MATCHING_TESTS\", \"NO_TESTS_COLLECTED\", \"UNAVAILABLE\", \"SKIPPED\", \"TIMEOUT\"\n    )\n    overall_ok = static_ok and tests_ok\n\n    verdict = (\n        \"PASS \u2014 Ready to confirm your inline bug repro and call submit_patch.\"\n        if overall_ok\n        else \"FAIL \u2014 DO NOT call submit_patch yet. Fix the errors/failures below and re-run verify_patch.\"\n    )\n\n    return {\n        \"verdict\": verdict,\n        \"ok\": overall_ok,\n        \"repo_root\": str(root),\n        \"source_files\": source_touched,\n        \"protected_files\": protected_touched,\n        \"untracked_files\": untracked,\n        \"syntax_errors\": syntax_errors,\n        \"matched_tests\": matched_tests,\n        \"pytest_status\": pytest_status,\n        \"filtered_preexisting\": filtered_preexisting,\n        \"pytest_output\": pytest_summary,\n        \"diff_stat\": diff_stat[:800],\n        \"warnings\": warnings,\n    }\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=\"Verify /workspace patch and run baseline-diff targeted pytest.\")\n    parser.add_argument(\"--root\", default=None, help=\"Optional explicit repository root (auto-detects cwd/.git first)\")\n    parser.add_argument(\"--no-pytest\", action=\"store_true\", help=\"Skip running pytest\")\n    args = parser.parse_args()\n    root = _resolve_repo_root(args.root)\n    report = verify_workspace(root, run_pytest=not args.no_pytest)\n    print(json.dumps(report, ensure_ascii=False, indent=2)[:4200])\n    return 0 if report[\"ok\"] else 1\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
  },
  "v9b_think512_verify_v6": {
    "agent.yaml": "name: gemma4_e4b_v9b_think512_verify_v6\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: Single-agent E4B think=512 private + verify-patch v6 (baseline-diff pytest, CLI exclusion, syntax context, loop guard).\ninstruction: !include prompts/system.md\nskills:\n  - skills/verify-patch\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\ngenerate_content_config: !include configs/sampling.yaml\n",
    "configs/sampling.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 4096\nthinking_config:\n  thinking_budget: 512\n  include_thoughts: false\n",
    "eval_config.yaml": "evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 60\n  max_time_minutes: 8\n  max_turns: 120\n",
    "prompts/system.md": "You are an autonomous software engineer fixing an issue in `/workspace`.\nYour core tools are `run_command`, `read_file`, `edit_file`, `write_file`, `get_status`, `submit_patch`, plus the `verify-patch` skill (`run_skill_script`). Always run shell commands via `run_command(command=...)`.\nGoal: make the smallest correct edit to SOURCE code in `/workspace` so the hidden test suite passes, verify with real tests + the `verify-patch` skill, then call `submit_patch`.\n\n## Workflow (Budget: 60 tool calls, 8 minutes)\n1. **Locate source fast (calls 1\u20136)**:\n   - Call `run_command` with `git grep -n -F -e '<Symbol1>' -e '<Symbol2>' -- '*.py' | head -25`.\n   - Call `read_file` on 80\u2013120 lines around the target implementation. For FastAPI doc-code issues, edit executable files under `docs_src/`.\n2. **Edit cleanly (by call 12 \u2014 reliable editing rules)**:\n   - Copy `old_string` **verbatim** from fresh `read_file` or `sed -n 'A,Bp' <file>` output (short 3\u201312 lines, exact indentation, NO manual `\\n`/`\\\"` escapes, `allow_multiple=false`).\n   - **After ANY single failed `edit_file` (including `old_string not found` OR missing parameter errors)**, DO NOT retry `edit_file` \u2014 immediately switch to a single-quoted Python heredoc via `run_command`:\n     ```bash\n     python3 - <<'EOF'\n     from pathlib import Path\n     p = Path(\"path/to/file.py\")\n     s = p.read_text()\n     old = \"\"\"exact old block\"\"\"\n     new = \"\"\"exact new block\"\"\"\n     assert s.count(old) == 1, f\"count={s.count(old)}\"\n     p.write_text(s.replace(old, new, 1))\n     EOF\n     ```\n   - NEVER use multiline `python3 -c \"...\"` (quotes break). ALWAYS use `python3 - <<'EOF' ... EOF`.\n   - NEVER call `write_file` on `/tmp/...` or to create scratch/test scripts.\n3. **Verify with `verify-patch` + real bug repro (REQUIRED before `submit_patch`)**:\n   - First, run an inline bug repro + 2 edge cases importing the real `/workspace` module via `python3 - <<'EOF' ... EOF` to confirm the reported bug is truly fixed.\n   - Next, call `run_skill_script(skill_name=\"verify-patch\", file_path=\"scripts/run_verify.py\")`. It automatically finds the REAL matching files in `tests/` (never guess test paths!), checks syntax/diff/protected files, and runs targeted `pytest`.\n   - **If `verify-patch` returns `VERDICT: FAIL` (or your bug repro fails)**: DO NOT call `submit_patch`. Read the traceback, fix the source code, and re-run your repro + `verify-patch` until `VERDICT: PASS`.\n   - If `verify-patch` reports a `syntax_error` you cannot fix in 1 edit, reset cleanly with `git checkout -- <file>` and re-apply your edit.\n   - NEVER call `verify-patch` twice in a row without editing source code in between (if `ok: true`, call `submit_patch`).\n4. **Submit cleanly**:\n   - Only after `verify-patch` reports `ok: true` (`VERDICT: PASS`), call `submit_patch` as your final tool action, confirm `patch_size > 0` and `files_changed > 0`, and write a 1-sentence summary.\n\n## Hard Rules\n- NEVER modify test files (`tests/`, `test_*.py`, `*_test.py`) or configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg`, `tox.ini`).\n- NEVER guess `pytest tests/...` paths without running `verify-patch` or `find tests -name ...`.\n- NEVER run bare `pytest` (`pytest .`), `pip install`, or `git commit`.\n- NEVER call `submit_patch` when `verify-patch` or your repro script just reported a failure (unless `budget_warning` forces an immediate final submit).\n- NEVER finish without calling `submit_patch` on a compiled, non-empty source fix.\n",
    "skills/verify-patch/SKILL.md": "---\nname: verify-patch\ndescription: Deterministic pre-submit verification gate that inspects git diff, checks Python syntax and protected files, auto-discovers matching real test files in tests/, runs baseline-diff targeted pytest (filtering pre-existing sandbox failures), and outputs a clear PASS/FAIL verdict.\n---\n\n# Verify Patch Skill\n\nRun `scripts/run_verify.py` via `run_skill_script(skill_name=\"verify-patch\", file_path=\"scripts/run_verify.py\")` after a source edit and right before `submit_patch`.\n\nIt deterministically verifies:\n1. Non-empty diff on source files (no protected test/config files modified, no stray untracked files).\n2. Zero Python syntax errors (`ast.parse`) across modified `.py` files (with line context & reset command if broken).\n3. Auto-discovers real `tests/` files matching your modified source files/symbols (skipping unrelated tutorial/CLI tests).\n4. Runs baseline-diff targeted `pytest`: filters out pre-existing sandbox failures so `FAIL` only triggers on real regressions introduced by your patch.\n",
    "skills/verify-patch/scripts/run_verify.py": "\"\"\"Deterministic pre-submit patch & baseline-diff targeted pytest verifier (v6).\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nPROTECTED_CONFIGS = {\n    \"conftest.py\", \"pytest.ini\", \"pyproject.toml\", \"tox.ini\",\n    \"setup.cfg\", \".pytest.ini\", \"sitecustomize.py\", \"usercustomize.py\",\n    \"_swegemma_stubs.py\",\n}\n\nGENERIC_DIRS = {\"src\", \"lib\", \"pkg\", \"docs_src\", \"\"}\nGENERIC_STEMS = {\"init\", \"main\", \"base\", \"types\", \"typing\"}\n\n\ndef is_cache_artifact(rel_path: str) -> bool:\n    p = Path(rel_path)\n    parts = set(p.parts)\n    if parts & {\"__pycache__\", \".pytest_cache\", \".mypy_cache\", \".ruff_cache\"}:\n        return True\n    if p.suffix in {\".pyc\", \".pyo\"}:\n        return True\n    return False\n\n\ndef is_protected(rel_path: str) -> bool:\n    p = Path(rel_path)\n    name = p.name\n    if name in PROTECTED_CONFIGS or p.suffix == \".pth\":\n        return True\n    parts_l = {part.lower() for part in p.parts}\n    if parts_l & {\"tests\", \"test\", \"testing\"} and p.suffix == \".py\":\n        return True\n    if name.startswith(\"test_\") and name.endswith(\".py\"):\n        return True\n    if name.endswith(\"_test.py\"):\n        return True\n    return False\n\n\ndef _run(\n    root: Path,\n    cmd: list[str],\n    timeout: int = 30,\n    extra_env: dict[str, str] | None = None,\n    text: bool = True,\n) -> subprocess.CompletedProcess:\n    env = None\n    if extra_env:\n        env = dict(os.environ)\n        env.update(extra_env)\n    return subprocess.run(\n        cmd,\n        cwd=root,\n        capture_output=True,\n        text=text,\n        timeout=timeout,\n        env=env,\n        check=False,\n    )\n\n\ndef _git_out(root: Path, *args: str) -> str:\n    return _run(root, [\"git\", \"-c\", \"safe.directory=*\", *args], timeout=15).stdout.strip()\n\n\ndef _resolve_repo_root(explicit_root: str | None) -> Path:\n    candidates: list[Path] = []\n    if explicit_root:\n        candidates.append(Path(explicit_root))\n    cwd = Path.cwd()\n    for anc in (cwd, *cwd.parents):\n        candidates.append(anc)\n        candidates.append(anc / \"workspace\")\n    candidates.append(Path(\"/workspace\"))\n\n    for cand in candidates:\n        if cand.is_dir() and (cand / \".git\").exists():\n            return cand\n    return cwd if cwd.is_dir() else Path(\"/workspace\")\n\n\ndef _resolve_pytest_cmd(root: Path) -> list[str]:\n    \"\"\"Prefer the sandbox's isolated venv (sibling to workspace/) so editable repo imports resolve.\"\"\"\n    venv_dirs = [root.parent / \"venv\", root / \".venv\", root / \"venv\"]\n    for vd in venv_dirs:\n        py_bin = vd / \"bin\" / \"pytest\"\n        if py_bin.is_file():\n            return [str(py_bin)]\n        py_exe = vd / \"bin\" / \"python\"\n        if py_exe.is_file():\n            return [str(py_exe), \"-m\", \"pytest\"]\n    pytest_bin = shutil.which(\"pytest\")\n    if pytest_bin:\n        return [pytest_bin]\n    return [sys.executable, \"-m\", \"pytest\"]\n\n\ndef _cache_path(root: Path) -> Path:\n    digest = hashlib.sha1(str(root.resolve()).encode(\"utf-8\", errors=\"ignore\")).hexdigest()[:12]\n    return Path(f\"/tmp/_swegemma_verify_cache_{digest}.json\")\n\n\ndef _load_cache(root: Path) -> dict:\n    p = _cache_path(root)\n    if p.is_file():\n        try:\n            data = json.loads(p.read_text(encoding=\"utf-8\"))\n            if isinstance(data, dict):\n                return data\n        except Exception:\n            pass\n    return {\"preexisting_failed\": [], \"last_diff_hash\": \"\", \"same_diff_streak\": 0}\n\n\ndef _save_cache(root: Path, data: dict) -> None:\n    try:\n        _cache_path(root).write_text(json.dumps(data), encoding=\"utf-8\")\n    except Exception:\n        pass\n\n\ndef _extract_changed_symbols(diff_text: str) -> list[str]:\n    syms: list[str] = []\n    seen: set[str] = set()\n    for line in diff_text.splitlines():\n        if line.startswith(\"@@\"):\n            m = re.search(r\"@@\\s+(?:def|async def|class)\\s+([A-Za-z_][A-Za-z0-9_]*)\", line)\n            if m and m.group(1) not in seen and len(m.group(1)) >= 4:\n                seen.add(m.group(1))\n                syms.append(m.group(1))\n        elif line.startswith((\"+\", \"-\")) and not line.startswith((\"+++\", \"---\")):\n            m = re.match(r\"^[+-]\\s*(?:def|async def|class)\\s+([A-Za-z_][A-Za-z0-9_]*)\", line)\n            if m and m.group(1) not in seen and len(m.group(1)) >= 4:\n                seen.add(m.group(1))\n                syms.append(m.group(1))\n    return syms[:8]\n\n\ndef _discover_test_files(root: Path, source_files: list[str], diff_text: str, max_files: int = 2) -> list[str]:\n    test_dirs = [d for d in (\"tests\", \"test\") if (root / d).is_dir()]\n    if not test_dirs:\n        return []\n\n    all_tests: list[Path] = []\n    for td in test_dirs:\n        for p in (root / td).rglob(\"*.py\"):\n            nm = p.name\n            if nm.startswith(\"test_\") or nm.endswith(\"_test.py\"):\n                all_tests.append(p.relative_to(root))\n\n    if not all_tests:\n        return []\n\n    stems: set[str] = set()\n    subpkg_tokens: set[str] = set()\n    is_cli_edit = any(\"cli\" in rel.lower() for rel in source_files)\n\n    for rel in source_files:\n        p = Path(rel)\n        if p.suffix != \".py\":\n            continue\n        clean_stem = p.stem.lower().lstrip(\"_\")\n        if clean_stem and clean_stem not in GENERIC_STEMS:\n            stems.add(clean_stem)\n        # Drop leading src/lib/pkg and the top-level package name itself (e.g. \"fastapi\", \"requests\")\n        # so generic tests like test_fastapi_cli.py don't match every file in the package.\n        pkg_dirs = [part.lower().lstrip(\"_\") for part in p.parts[:-1] if part.lower().lstrip(\"_\") not in GENERIC_DIRS]\n        for subpkg in pkg_dirs[1:]:\n            subpkg_tokens.add(subpkg)\n\n    sym_hits: dict[str, int] = {}\n    symbols = _extract_changed_symbols(diff_text)\n    if symbols:\n        grep_cmd = [\"git\", \"-c\", \"safe.directory=*\", \"grep\", \"-l\", \"-F\"]\n        for sym in symbols:\n            grep_cmd.extend([\"-e\", sym])\n        grep_cmd.append(\"--\")\n        grep_cmd.extend(test_dirs)\n        proc = _run(root, grep_cmd, timeout=10)\n        for line in proc.stdout.splitlines():\n            line = line.strip()\n            if line:\n                sym_hits[line] = sym_hits.get(line, 0) + 1\n\n    scored: list[tuple[int, int, str]] = []\n    for tp in all_tests:\n        rel_str = tp.as_posix()\n        name_l = tp.name.lower()\n        path_l = rel_str.lower()\n        # Skip tutorial/doc example tests & unrelated CLI tests unless editing CLI code\n        if \"tutorial\" in path_l:\n            continue\n        if not is_cli_edit and (\"_cli\" in name_l or \"cli_\" in name_l or \"/cli/\" in path_l):\n            continue\n        score = 0\n\n        for st in stems:\n            if name_l in (f\"test_{st}.py\", f\"{st}_test.py\"):\n                score += 120\n            elif f\"_{st}.\" in name_l or f\"_{st}_\" in name_l or name_l.startswith(f\"test_{st}_\"):\n                score += 70\n            elif st in name_l:\n                score += 40\n            if f\"/{st}/\" in path_l:\n                score += 30\n\n        for pt in subpkg_tokens:\n            if pt in name_l or f\"test_{pt}\" in path_l or f\"/{pt}/\" in path_l:\n                score += 35\n\n        if rel_str in sym_hits:\n            score += 45 * sym_hits[rel_str]\n\n        if score > 0:\n            scored.append((score, -len(rel_str), rel_str))\n\n    scored.sort(reverse=True)\n    return [item[2] for item in scored[:max_files]]\n\n\ndef _extract_failed_nodes(output: str) -> list[str]:\n    nodes: list[str] = []\n    seen: set[str] = set()\n    for line in output.splitlines():\n        m = re.match(r\"^FAILED\\s+(.+?)(?:\\s+-\\s+.*)?$\", line.strip())\n        if m:\n            node = m.group(1).strip()\n            if \"::\" in node and node not in seen:\n                seen.add(node)\n                nodes.append(node)\n    return nodes[:8]\n\n\ndef _check_baseline_failures(\n    root: Path,\n    base_ref: str,\n    source_touched: list[str],\n    failed_nodes: list[str],\n    pytest_cmd: list[str],\n) -> set[str]:\n    \"\"\"Run ONLY the failed node IDs against base_ref to separate pre-existing sandbox failures from new regressions.\"\"\"\n    if not failed_nodes or not source_touched:\n        return set()\n\n    saved_bytes: dict[Path, bytes] = {}\n    baseline_bytes: dict[Path, bytes] = {}\n\n    for rel in source_touched:\n        full = root / rel\n        if not full.is_file():\n            continue\n        # Only temporarily revert files that exist in base_ref\n        chk = _run(root, [\"git\", \"-c\", \"safe.directory=*\", \"cat-file\", \"-e\", f\"{base_ref}:{rel}\"], timeout=10)\n        if chk.returncode != 0:\n            continue\n        show = _run(root, [\"git\", \"-c\", \"safe.directory=*\", \"show\", f\"{base_ref}:{rel}\"], timeout=10, text=False)\n        if show.returncode != 0:\n            continue\n        saved_bytes[full] = full.read_bytes()\n        baseline_bytes[full] = show.stdout\n\n    if not saved_bytes:\n        return set()\n\n    try:\n        for full, bdata in baseline_bytes.items():\n            full.write_bytes(bdata)\n        cmd = [\n            *pytest_cmd,\n            *failed_nodes,\n            \"-q\", \"--tb=no\", \"--no-header\",\n            \"--continue-on-collection-errors\",\n            \"-o\", \"cache_dir=/tmp/pytest_cache\",\n        ]\n        proc = _run(root, cmd, timeout=25, extra_env={\"PYTHONDONTWRITEBYTECODE\": \"1\"})\n        combined = ((proc.stdout or \"\") + \"\\n\" + (proc.stderr or \"\")).strip()\n        return set(_extract_failed_nodes(combined))\n    except subprocess.TimeoutExpired:\n        return set()\n    finally:\n        for full, orig in saved_bytes.items():\n            full.write_bytes(orig)\n\n\ndef _run_targeted_pytest(\n    root: Path,\n    pytest_cmd: list[str],\n    matched_tests: list[str],\n    deselect_nodes: list[str],\n) -> tuple[int, str]:\n    flags = [\n        \"--maxfail=5\", \"-q\", \"--tb=short\", \"--no-header\",\n        \"--continue-on-collection-errors\",\n        \"-o\", \"cache_dir=/tmp/pytest_cache\",\n    ]\n    for node in deselect_nodes:\n        file_part = node.split(\"::\", 1)[0]\n        if file_part in matched_tests:\n            flags.extend([\"--deselect\", node])\n    cmd = [*pytest_cmd, *matched_tests, *flags]\n    proc = _run(root, cmd, timeout=60, extra_env={\"PYTHONDONTWRITEBYTECODE\": \"1\"})\n    combined = ((proc.stdout or \"\") + \"\\n\" + (proc.stderr or \"\")).strip()\n    return proc.returncode, combined\n\n\ndef _syntax_context(full: Path, lineno: int | None) -> str:\n    if not lineno or not full.is_file():\n        return \"\"\n    lines = full.read_text(encoding=\"utf-8\", errors=\"ignore\").splitlines()\n    start = max(1, lineno - 2)\n    end = min(len(lines), lineno + 2)\n    return \"\\n\".join(\n        f\"{'>>' if i == lineno else '  '} {i}: {lines[i - 1]}\"\n        for i in range(start, end + 1)\n    )\n\n\ndef verify_workspace(root: Path, run_pytest: bool = True) -> dict:\n    has_baseline = bool(_git_out(root, \"rev-parse\", \"--verify\", \"_swegemma_baseline\"))\n    base_ref = \"_swegemma_baseline\" if has_baseline else \"HEAD\"\n\n    changed = [\n        line for line in _git_out(root, \"diff\", \"--name-only\", base_ref).splitlines()\n        if line.strip() and not is_cache_artifact(line)\n    ]\n    untracked = [\n        line for line in _git_out(root, \"ls-files\", \"--others\", \"--exclude-standard\").splitlines()\n        if line.strip() and not is_cache_artifact(line)\n    ]\n    all_touched = sorted(set(changed) | set(untracked))\n\n    protected_touched = [p for p in all_touched if is_protected(p)]\n    source_touched = [p for p in all_touched if not is_protected(p)]\n\n    syntax_errors = []\n    for rel in source_touched:\n        full = root / rel\n        if full.suffix == \".py\" and full.is_file():\n            try:\n                ast.parse(full.read_text(encoding=\"utf-8\", errors=\"ignore\"), filename=rel)\n            except SyntaxError as exc:\n                syntax_errors.append({\n                    \"file\": rel,\n                    \"line\": exc.lineno,\n                    \"message\": exc.msg,\n                    \"context\": _syntax_context(full, exc.lineno),\n                })\n\n    diff_text = _git_out(root, \"diff\", \"-U0\", base_ref, \"--\", *source_touched) if source_touched else \"\"\n    diff_stat = _git_out(root, \"diff\", \"--stat\", base_ref)\n\n    cache = _load_cache(root)\n    preexisting_failed: list[str] = list(cache.get(\"preexisting_failed\") or [])\n    diff_hash = hashlib.sha256(diff_text.encode(\"utf-8\", errors=\"ignore\")).hexdigest()[:12]\n    same_diff_streak = (cache.get(\"same_diff_streak\", 0) + 1) if (diff_hash and diff_hash == cache.get(\"last_diff_hash\")) else 0\n\n    warnings: list[str] = []\n    if not source_touched:\n        warnings.append(\"No source files modified yet; submit_patch would be empty.\")\n    if protected_touched:\n        revert_cmd = \"git checkout -- \" + \" \".join(protected_touched)\n        warnings.append(f\"Protected test/config files touched (revert immediately with `{revert_cmd}`): {protected_touched}\")\n    if untracked:\n        warnings.append(f\"Untracked files in /workspace (remove scratch files): {untracked}\")\n    if syntax_errors:\n        bad_files = \" \".join(sorted({e[\"file\"] for e in syntax_errors}))\n        warnings.append(\n            f\"Python syntax error(s) detected. Inspect syntax_errors[].context and fix, \"\n            f\"or reset cleanly with `git checkout -- {bad_files}` and re-apply your edit.\"\n        )\n    if same_diff_streak >= 1:\n        warnings.append(\n            \"LOOP GUARD: Workspace diff has NOT changed since your previous verify-patch call. \"\n            \"Do NOT call verify-patch again without editing source code first (or call submit_patch if verdict is PASS).\"\n        )\n\n    matched_tests: list[str] = []\n    pytest_status = \"SKIPPED\"\n    pytest_summary = \"\"\n    filtered_preexisting: list[str] = []\n\n    if run_pytest and source_touched and not syntax_errors:\n        matched_tests = _discover_test_files(root, source_touched, diff_text, max_files=2)\n        if not matched_tests:\n            pytest_status = \"NO_MATCHING_TESTS\"\n            pytest_summary = \"No matching test file found under tests/; run an inline python3 heredoc repro instead.\"\n        else:\n            pytest_cmd = _resolve_pytest_cmd(root)\n            try:\n                for attempt in range(2):\n                    rc, combined = _run_targeted_pytest(root, pytest_cmd, matched_tests, preexisting_failed)\n                    has_real_failure = (\n                        \"=== FAILURES ===\" in combined\n                        or bool(re.search(r\"\\b\\d+\\s+failed\\b\", combined))\n                    )\n                    has_env_error = (\n                        \"=== ERRORS ===\" in combined\n                        or \"ERROR collecting\" in combined\n                        or \"ERROR at setup\" in combined\n                    )\n                    if rc == 0:\n                        pytest_status = \"PASS\"\n                        lines = [ln for ln in combined.splitlines() if ln.strip()]\n                        pytest_summary = \"\\n\".join(lines[-5:])\n                        break\n                    if rc == 5:\n                        pytest_status = \"NO_TESTS_COLLECTED\"\n                        pytest_summary = combined[-600:]\n                        break\n                    if \"No module named pytest\" in combined or \"disabled\" in combined.lower():\n                        pytest_status = \"UNAVAILABLE\"\n                        pytest_summary = \"pytest unavailable in sandbox; rely on real-import python3 heredoc check.\"\n                        break\n                    if has_real_failure:\n                        failed_nodes = _extract_failed_nodes(combined)\n                        base_failed = _check_baseline_failures(root, base_ref, source_touched, failed_nodes, pytest_cmd)\n                        new_regressions = [n for n in failed_nodes if n not in base_failed]\n                        if base_failed:\n                            for bf in sorted(base_failed):\n                                if bf not in preexisting_failed:\n                                    preexisting_failed.append(bf)\n                                if bf not in filtered_preexisting:\n                                    filtered_preexisting.append(bf)\n                        if not new_regressions and failed_nodes:\n                            # Every observed failure already existed on baseline!\n                            if attempt == 0:\n                                # Re-run once with --deselect so tests AFTER the pre-existing failure still execute.\n                                continue\n                            pytest_status = \"PASS_PREEXISTING_ONLY\"\n                            pytest_summary = (\n                                f\"Filtered pre-existing sandbox failure(s) that already fail on baseline: {filtered_preexisting}. \"\n                                \"No new regressions detected.\"\n                            )\n                            break\n                        pytest_status = \"FAIL\"\n                        pytest_summary = combined[-1800:]\n                        warnings.append(\n                            f\"CONFIRMED NEW REGRESSION(S) in {matched_tests}: {new_regressions or failed_nodes}. \"\n                            \"Inspect traceback and fix source before submitting.\"\n                        )\n                        break\n                    if has_env_error:\n                        pytest_status = \"ENV_SKIPPED\"\n                        pytest_summary = (\n                            \"Only sandbox collection/fixture setup errors occurred (0 test assertions failed). \"\n                            \"Rely on your inline python3 heredoc repro to verify the fix.\"\n                        )\n                        break\n                    pytest_status = \"FAIL\"\n                    pytest_summary = combined[-1800:]\n                    warnings.append(f\"Targeted pytest exited {rc} on {matched_tests}; inspect output before submitting.\")\n                    break\n            except subprocess.TimeoutExpired:\n                pytest_status = \"TIMEOUT\"\n                pytest_summary = f\"pytest timed out after 60s on {matched_tests}; run a narrower `-k` test filter.\"\n\n    cache[\"preexisting_failed\"] = preexisting_failed[:20]\n    cache[\"last_diff_hash\"] = diff_hash\n    cache[\"same_diff_streak\"] = same_diff_streak\n    _save_cache(root, cache)\n\n    if filtered_preexisting and pytest_status == \"PASS\":\n        pytest_summary = (\n            f\"{pytest_summary}\\n(Note: deselected pre-existing baseline failure(s): {filtered_preexisting})\"\n        ).strip()\n\n    static_ok = bool(source_touched) and not protected_touched and not syntax_errors\n    tests_ok = pytest_status in (\n        \"PASS\", \"PASS_PREEXISTING_ONLY\", \"ENV_SKIPPED\", \"NO_MATCHING_TESTS\", \"NO_TESTS_COLLECTED\", \"UNAVAILABLE\", \"SKIPPED\", \"TIMEOUT\"\n    )\n    overall_ok = static_ok and tests_ok\n\n    verdict = (\n        \"PASS \u2014 Ready to confirm your inline bug repro and call submit_patch.\"\n        if overall_ok\n        else \"FAIL \u2014 DO NOT call submit_patch yet. Fix the errors/failures below and re-run verify_patch.\"\n    )\n\n    return {\n        \"verdict\": verdict,\n        \"ok\": overall_ok,\n        \"repo_root\": str(root),\n        \"source_files\": source_touched,\n        \"protected_files\": protected_touched,\n        \"untracked_files\": untracked,\n        \"syntax_errors\": syntax_errors,\n        \"matched_tests\": matched_tests,\n        \"pytest_status\": pytest_status,\n        \"filtered_preexisting\": filtered_preexisting,\n        \"pytest_output\": pytest_summary,\n        \"diff_stat\": diff_stat[:800],\n        \"warnings\": warnings,\n    }\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=\"Verify /workspace patch and run baseline-diff targeted pytest.\")\n    parser.add_argument(\"--root\", default=None, help=\"Optional explicit repository root (auto-detects cwd/.git first)\")\n    parser.add_argument(\"--no-pytest\", action=\"store_true\", help=\"Skip running pytest\")\n    args = parser.parse_args()\n    root = _resolve_repo_root(args.root)\n    report = verify_workspace(root, run_pytest=not args.no_pytest)\n    print(json.dumps(report, ensure_ascii=False, indent=2)[:4200])\n    return 0 if report[\"ok\"] else 1\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
  }
}
tasks = load_tasks(TASKS_PATH)
wanted = set(SOUND12_IDS)
subset = sorted([t for t in tasks if t.instance_id in wanted], key=lambda t: t.instance_id)
assert len(subset) == len(SOUND12_IDS), f'Expected {len(SOUND12_IDS)}, got {len(subset)}'

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=False,
    max_loras=1,
    max_lora_rank=128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=(),
)
server_instance = VllmServer(vllm_cfg)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})', flush=True)

models = server_instance.create_model_registry(
    aliases=[DECLARED_MODEL],
    model_prefix='openai/',
    api_key=os.environ.get('LOCAL_API_KEY', 'EMPTY'),
)

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if (callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn)) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

limits, gen_constraints = build_submission_limits()
all_summaries = []

for run_name, files_map in VARIANTS.items():
    print(f'\n================ RUNNING {run_name} ({len(subset)} tasks) ================', flush=True)
    run_dir = WORKING_DIR / run_name
    agent_dir = run_dir / 'agent'
    if agent_dir.exists():
        shutil.rmtree(agent_dir)
    for rel, content in files_map.items():
        p = agent_dir / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding='utf-8')

    ev_raw = yaml.safe_load((agent_dir / 'eval_config.yaml').read_text(encoding='utf-8')) or {}
    ev = ev_raw.get('evaluation', ev_raw)
    max_time_minutes = float(ev.get('max_time_minutes', 8.0))
    adapters = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

    eval_config = EvalConfig(
        tasks_path=TASKS_PATH,
        snapshots_dir=DATA_DIR / 'snapshots',
        results_dir=run_dir / 'results',
        submission_dir=agent_dir,
        models=models,
        sandbox='subprocess',
        timeout_seconds=int(ev.get('timeout_seconds', 240)),
        verification_timeout_seconds=int(ev.get('verification_timeout_seconds', 300)),
        max_time_minutes=max_time_minutes,
        max_tool_calls=int(ev.get('max_tool_calls', 28)),
        max_turns=int(ev.get('max_turns', 80)),
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=GRAPH_DIR,
        embeddings_dir=EMBEDDINGS_DIR,
        wheels_dir=DATA_DIR / 'wheels',
        verbose=False,
    )
    evaluator = Evaluator(eval_config)
    rows_path = run_dir / 'per_task.jsonl'
    rows = []
    for idx, task in enumerate(subset, start=1):
        started = time.perf_counter()
        try:
            result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(subset))
            patch = (result.agent_patch or '').strip()
            row = dict(
                run=run_name, id=task.instance_id, repo=task.repo,
                resolved=bool(result.resolved), patch_chars=len(patch),
                tool_calls=result.tool_calls,
                agent_seconds=round(float(result.duration_seconds or 0.0), 1),
                test_exit_code=result.test_exit_code,
                error=getattr(result, 'error', None),
            )
        except Exception as exc:
            row = dict(
                run=run_name, id=task.instance_id, repo=task.repo,
                resolved=False, patch_chars=0, tool_calls=None,
                agent_seconds=None, test_exit_code=None, error=repr(exc)[:300],
            )
        row['wall_seconds'] = round(time.perf_counter() - started, 1)
        rows.append(row)
        with rows_path.open('a') as f:
            f.write(json.dumps(row) + '\n')
        print(f"[{run_name} {idx}/{len(subset)}] {row['id']}: resolved={row['resolved']} "
              f"patch={row['patch_chars']} calls={row['tool_calls']} "
              f"agent={row['agent_seconds']}s wall={row['wall_seconds']}s err={row['error']}", flush=True)

    df = pd.DataFrame(rows)
    agent_s = df['agent_seconds'].dropna()
    summary = {
        'run': run_name,
        'tasks': len(df),
        'resolved': int(df['resolved'].sum()),
        'resolution_rate': round(float(df['resolved'].mean()), 3),
        'no_patch': int((df['patch_chars'] == 0).sum()),
        'errors': int(df['error'].notna().sum()),
        'mean_tool_calls': round(float(df['tool_calls'].dropna().mean()), 1) if len(df['tool_calls'].dropna()) else None,
        'agent_seconds_median': float(agent_s.median()) if len(agent_s) else None,
        'wall_seconds_mean': round(float(df['wall_seconds'].mean()), 1),
    }
    (run_dir / 'summary.json').write_text(json.dumps(summary, indent=2))
    all_summaries.append(summary)
    print(json.dumps(summary, indent=2), flush=True)
    print(df.to_string(index=False), flush=True)

(WORKING_DIR / 'ablation_summary.json').write_text(json.dumps(all_summaries, indent=2))
print('\n=== FINAL ABLATION COMPARISON ===', flush=True)
print(pd.DataFrame(all_summaries).to_string(index=False), flush=True)
